***

##  MLOPS - PRACTICAL 5

***

## aim:

To monitor the performance of a trained machine-learning model on new production data and detect performance degradation using MLflow.

## problem statement:

Load a trained bank loan-default model, evaluate it on new loan applications, calculate production accuracy, log the model parameters and metric using MLflow, and recommend retraining when accuracy falls below `0.80`.

***

## python code:

### 1. model schema check

In [2]:
import joblib
import pandas as pd

model = joblib.load("loan_default_model.pkl")

print("Model type:", type(model).__name__)
print("\nNumber of features expected:", model.n_features_in_)
print("\nFeature names expected:", list(model.feature_names_in_))

new_data = pd.read_csv("new_loan_applications.csv")

expected = set(model.feature_names_in_)
available = set(new_data.columns)

missing = expected - available

if missing:
    print("\nMISMATCH: dataset is missing required features:", missing)
else:
    print("\nOK: dataset contains all features the model expects.")

Model type: RandomForestClassifier

Number of features expected: 8

Feature names expected: ['Age', 'Annual_Income', 'Loan_Amount', 'Credit_Score', 'Employment_Years', 'Existing_Loans', 'Debt_to_Income_Ratio', 'Monthly_Expenses']

OK: dataset contains all features the model expects.


C:\Users\Shinde\AppData\Roaming\Python\Python311\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.9.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\Shinde\AppData\Roaming\Python\Python311\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.9.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


### 2. model drift monitoring

In [5]:
import pandas as pd
import joblib
import mlflow
from sklearn.metrics import accuracy_score

# Load trained model
model = joblib.load("loan_default_model.pkl")
print("Model loaded successfully!")

# Load new production data
new_data = pd.read_csv("new_loan_applications.csv")
print("New production data loaded successfully!")

# Select ML features
features = [
    "Age",
    "Annual_Income",
    "Loan_Amount",
    "Credit_Score",
    "Employment_Years",
    "Existing_Loans",
    "Debt_to_Income_Ratio",
    "Monthly_Expenses"
]

X_new = new_data[features]
y_new = new_data["Loan_Default"]

# Make predictions
predictions = model.predict(X_new)

# Calculate accuracy
accuracy = accuracy_score(y_new, predictions)

print("-----------------------------------")
print("MODEL DRIFT MONITORING")
print("-----------------------------------")
print("Production Accuracy:", round(accuracy, 2))

# MLflow experiment
mlflow.set_experiment("Loan_Default_Model_Monitoring")

with mlflow.start_run():

    mlflow.log_param("model", "Random Forest")
    mlflow.log_param("n_estimators", 100)
    mlflow.log_metric("production_accuracy", accuracy)

    print("Results logged successfully to MLflow")

# Performance threshold
threshold = 0.80

if accuracy < threshold:
    print()
    print("WARNING: Model performance has degraded!")
    print("Model retraining is recommended.")
else:
    print()
    print("Model performance is stable.")

C:\Users\Shinde\AppData\Roaming\Python\Python311\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.9.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\Shinde\AppData\Roaming\Python\Python311\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.9.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


Model loaded successfully!
New production data loaded successfully!
-----------------------------------
MODEL DRIFT MONITORING
-----------------------------------
Production Accuracy: 0.65


2026/08/30 22:37:46 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/08/30 22:37:46 INFO mlflow.store.db.utils: Updating database tables
2026/08/30 22:37:47 INFO mlflow.tracking.fluent: Experiment with name 'Loan_Default_Model_Monitoring' does not exist. Creating a new experiment.


Results logged successfully to MLflow

Model retraining is recommended.


***

## theory:

### - model drift

Model drift occurs when the performance or behaviour of a deployed model changes because the real-world environment or incoming data has changed.

### - data drift

A change in the distribution of input features.

### - concept drift

A change in the relationship between input features and the target.

### - mlflow

MLflow is used here to track the model-monitoring experiment, parameters and production-performance metric.

### - parameter

A configuration/input value, such as,  n_estimators = 100


### - metric

A measured result, such as,  production_accuracy = 0.65

### - threshold

The minimum acceptable performance level.  Here, threshold = 0.80

***

## output:

The supplied practical demonstrates, Production Accuracy: 0.65

Since, 0.65 < 0.80

the program prints:

```text
WARNING: Model performance has degraded!
Model retraining is recommended.
```

MLflow records:

- `model = Random Forest`
- `n_estimators = 100`
- `production_accuracy`

***

## viva questions:

1. **What is model drift?**  
   A change in deployed model performance or behaviour caused by changes in incoming data or the real-world environment.

2. **What is data drift?**  
   A change in the distribution of input features.

3. **What is concept drift?**  
   A change in the relationship between inputs and the target.

4. **Why monitor a deployed model?**  
   Good training/test performance does not guarantee good performance on future production data.

5. **What is MLflow used for here?**  
   Tracking the experiment, parameters and production-performance metric.

6. **What is an MLflow experiment?**  
   A logical group of related runs.

7. **What is an MLflow run?**  
   One tracked execution within an experiment.

8. **What is `log_param()`?**  
   It records a parameter/configuration value.

9. **What is `log_metric()`?**  
   It records a measured metric.

10. **Why is `accuracy_score()` used?**  
    To compare the production predictions with the actual labels.

11. **Why is the target column required for accuracy?**  
    Accuracy requires actual labels to compare with predictions.

12. **What happens if accuracy is below 0.80?**  
    The model is flagged as degraded and retraining is recommended.

13. **Why check `model.feature_names_in_`?**  
    To verify that the incoming data contains the features expected by the trained model.

14. **Why use `joblib.load()`?**  
    To load the previously trained serialized model.

15. **Why might retraining be necessary?**  
    Because the patterns learned during training may no longer represent current production data.

***

## result:

The trained loan-default model is evaluated on new production data. MLflow records the monitoring information, and a drop in production accuracy below the defined threshold triggers a model-degradation warning and retraining recommendation.

## observation:

The trained bank loan-default model was evaluated on new production data. The production accuracy is compared with the defined threshold and the monitoring information is recorded in MLflow. When production performance falls below the threshold, the model is flagged as degraded and retraining is recommended.

***